[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/01_agente_base/01_agente_base.ipynb)

# Módulo 01 · El agente base: SupportOps v0, proveedor agnóstico, tráfico y fallos

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración estimada** | ≈ 4 h 20 min (lección ≈ 2 h 30 min + proyecto ≈ 1 h 50 min) |
| **Servicios de Compose** | **Ninguno** (todo es simulado y corre en Python). Funciona igual en Colab |
| **RAM aproximada** | < 300 MB (solo el kernel de Python) |
| **Prerrequisitos** | Módulo 00 (mapa del stack, «Docker listo ≠ arrancado», las cinco grietas del monitoreo clásico) |
| **Hilo conductor** | Aquí nace **SupportOps v0**: el agente de soporte que instrumentarás a partir del módulo 03 |

## Objetivos de aprendizaje

Al terminar esta lección podrás:

1. **Definir** qué es un agente (bucle LLM + herramientas) y distinguirlo de una llamada simple y de un *workflow*.
2. **Implementar a mano** el bucle de un agente en ≈ 25 líneas y **leer** el de `SupportAgent`.
3. **Calcular** cómo crecen los tokens de entrada y el coste con el número de pasos (efecto cuadrático del contexto).
4. **Cambiar de proveedor LLM** (`mock | anthropic | openai | ollama`) solo con variables de entorno, sin tocar el agente.
5. **Describir** la base de conocimiento (RAG con TF-IDF) y las herramientas de SupportOps, y **distinguir** error de negocio de error de infraestructura.
6. **Generar** tráfico sintético reproducible (semilla, esquema, ventanas de incidente) y **ejecutarlo** a través del agente.
7. **Inyectar** siete tipos de fallo y **explicar** qué huella deja cada uno en latencia, tokens, estado y calidad.
8. **Identificar** los puntos de extensión donde se instrumentará el agente (módulo 03).

> **🧭 Ruta · Ruta núcleo vs ampliación**
>
> Este módulo tiene ≈ 75 celdas y ≈ 4 h 20 min: puedes hacerlo en **dos sesiones**.
>
> - **Ruta núcleo (≈ 2 h 15 min, imprescindible para el módulo 03):** §1 (agente y bucle a mano), §1.5 (parada y coste), §3.1 (herramientas; error de negocio ≠ de infraestructura), §4 (tráfico), §5 (fallos inyectados), §7 (puntos de extensión) y el proyecto.
> - **Ampliación (≈ 1 h, puedes volver más tarde):** §2 (adaptadores de proveedor y modelo de latencia del mock), §3.2 (RAG con TF-IDF y su fórmula), §6 (experimento sano vs con fallos) y los retos extra del proyecto.
> - **Si nunca has visto un agente:** lee la sección 1.3 con el bucle a mano *ejecutándola línea a línea* antes de seguir; todo lo demás se apoya en ese `for`.

## 🔁 Conexión con el módulo anterior

Escribe (o piensa) tu respuesta **antes** de abrirla: recuperar de memoria es lo que fija el aprendizaje, y estas tres ideas las usarás en todo el módulo.

<details>
<summary>1 · (Diagnóstico) Acabas de lanzar `docker compose --profile core up -d`; `docker compose ps` dice «Up» para Loki pero tu primer log no aparece. ¿Qué hipótesis tienes y qué compruebas?</summary>

Puede que Loki esté *arrancado* pero **no listo**: tardó ≈ 17 s en responder `200` en `/ready` (Pyroscope ≈ 1 minuto). Compruebas el readiness (`check_stack()` o `curl :3100/ready`) y, de extremo a extremo, una señal canario buscada por su `trace_id`.

</details>

<details>
<summary>2 · (Transferencia) Tu panel de «% de peticiones sin error» marca 99,5 % pero los clientes se quejan de respuestas inventadas. ¿Qué «grieta» del monitoreo clásico es y qué dos de las otras cuatro conoces?</summary>

Es la grieta 4: **«200 OK» ya no significa éxito**. Las otras: latencia de cola pesada (percentiles, no medias), cadena de longitud variable (bucles), coste ∝ tokens y concentrado, y fallos parciales silenciosos (herramienta caída + respuesta de cortesía). Aquí en el módulo 01 las verás una a una como fallos inyectados.

</details>

<details>
<summary>3 · (Cálculo) En el diagrama del stack, ¿qué componentes *escriben* y cuáles *leen* telemetría? Si Grafana se apaga una hora, ¿pierdes datos?</summary>

Escriben: la app → Collector → Tempo/Loki/Prometheus. Leen: Grafana (flechas discontinuas). Con Grafana apagado **no pierdes datos**, solo la vista; si cae el Collector sí hay riesgo de pérdida porque es el único punto de salida de la app.

</details>

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

_ = check_stack([])   # este módulo NO necesita servicios: solo muestra el estado por si quieres tenerlos levantados

---
# 1 · ¿Qué es un agente?

## 1.1 Intuición: quién decide el siguiente paso

Anthropic, en su guía [«Building effective agents»](https://www.anthropic.com/engineering/building-effective-agents) (dic. 2024), distingue dos tipos de «sistemas agénticos»:

- **Workflows**: *LLMs y herramientas orquestados por caminos de código predefinidos* — el programador decide el orden.
- **Agentes**: *sistemas donde el LLM dirige dinámicamente su propio proceso y el uso de herramientas* — el modelo decide el orden.

Y resume su implementación así: suelen ser **«LLMs usando herramientas según el feedback del entorno, en un bucle»**; además es habitual incluir **condiciones de parada** (p. ej. un número máximo de iteraciones).
La misma guía avisa de que la autonomía implica **mayor coste y posibilidad de errores acumulados**. Esas dos frases son, literalmente, el motivo de este curso.

| Si vienes de… | Analogía |
|---|---|
| **ML** | Un *pipeline* (workflow) es un `Pipeline.fit/predict` de sklearn con pasos fijos; un agente es una **política** que, en cada estado, elige la siguiente acción (como en aprendizaje por refuerzo, pero con un LLM como política y sin entrenamiento) |
| **Backend** | Un workflow es un orquestador con un DAG fijo; un agente es un *servicio que decide a quién llamar en tiempo de ejecución* |
| **SRE** | El grafo de dependencias del servicio **no está en el código**: cambia con cada petición, así que el *runbook* clásico («si cae X, mira Y») no basta |

In [ ]:
fig, ax = viz.canvas(12, 4.6, (0, 100), (0, 40))
cols = [(2, "1 · Llamada al LLM", viz.PALETTE["neutral"]), (35, "2 · Workflow\n(el código decide)", viz.PALETTE["logs"]), (68, "3 · Agente\n(el LLM decide)", viz.PALETTE["traces"])]
for x, title, c in cols:
    ax.text(x + 14, 37, title, ha="center", va="top", fontsize=11, fontweight="bold", color=c)
# 1
a = viz.box(ax, 4, 16, 10, 6, "Entrada", c := viz.PALETTE["neutral"]); b = viz.box(ax, 18, 16, 10, 6, "LLM", c, bold=True)
viz.arrow(ax, viz.right(a), viz.left(b)); ax.text(16, 10, "una sola llamada;\nsin herramientas", ha="center", fontsize=9)
# 2
c = viz.PALETTE["logs"]; w1 = viz.box(ax, 36, 17, 10, 6, "LLM:\nclasifica", c); w2 = viz.box(ax, 52, 24, 12, 5, "if pedido:", c, fontsize=8)
w3 = viz.box(ax, 52, 16, 12, 5, "get_order()", c, fontsize=8); w4 = viz.box(ax, 52, 8, 12, 5, "search_kb()", c, fontsize=8)
viz.arrow(ax, viz.right(w1), viz.left(w2)); viz.arrow(ax, viz.right(w1), viz.left(w3)); viz.arrow(ax, viz.right(w1), viz.left(w4))
ax.text(49, 1.5, "caminos fijos en el código;\nel LLM rellena huecos", ha="center", fontsize=9)
# 3
c = viz.PALETTE["traces"]; g1 = viz.box(ax, 70, 20, 11, 8, "LLM", c, bold=True); g2 = viz.box(ax, 87, 26, 11, 5, "tool A", c, fontsize=8); g3 = viz.box(ax, 87, 18, 11, 5, "tool B", c, fontsize=8)
g4 = viz.box(ax, 87, 10, 11, 5, "respuesta", c, fontsize=8)
viz.arrow(ax, viz.right(g1), viz.left(g2)); viz.arrow(ax, viz.right(g1), viz.left(g3)); viz.arrow(ax, viz.right(g1), viz.left(g4))
viz.arrow(ax, (g2[0] + 5, g2[1] + 6), (g1[0] + 5, g1[1] + 8.6), "", c, rad=0.5, dashed=True); viz.arrow(ax, (g3[0], g3[1] + 5.2), (g1[0] + 11.4, g1[1] + 1), "", c, rad=-0.3, dashed=True)
ax.text(83, 1.5, "el LLM elige qué herramienta y\ncuántas veces: bucle", ha="center", fontsize=9)
plt.show()

> 👀 **Qué debes observar**
>
> - De izquierda a derecha **aumenta la autonomía… y la incertidumbre**: en 1 hay un único punto de fallo; en 2 el grafo de llamadas es *conocido* (puedes dibujarlo en tiempo de diseño); en 3 el grafo **se decide en ejecución**.
> - La observabilidad de un agente (3) tiene que *reconstruir* ese grafo a partir de lo que ocurrió — por eso necesitaremos trazas. Para (1) y (2) bastaban métricas y logs clásicos.
> - Regla práctica de la guía de Anthropic: usa la solución **más simple** que funcione; los agentes intercambian latencia y coste por mejor rendimiento en la tarea.

## 1.2 Una llamada simple al LLM (nivel 1)

Usamos la capa de proveedor del curso. Por defecto es el **modo `mock`**: un LLM simulado, determinista y sin coste (lo estudiamos en la sección 2).
Una llamada recibe mensajes y devuelve una respuesta **normalizada**.

In [ ]:
llm = get_llm()                                   # LLM_PROVIDER=mock → MockLLM("mock-small")
resp = llm.chat([Message("user", "¿Cuántos años de garantía tienen los productos?")], system="Eres un asistente de soporte.",
                call_key=("demo", 0))
print("proveedor/modelo :", resp.provider, "/", resp.model)
print("finish_reason     :", resp.finish_reason, "(¿pide herramientas?:", bool(resp.tool_calls), ")")
print("tokens (in/out)   :", resp.usage.input_tokens, "/", resp.usage.output_tokens)
print("latencia simulada :", resp.latency_s, "s")
print("contenido         :", repr(resp.content[:80]))
print("tool_calls        :", [(c.name, c.arguments) for c in resp.tool_calls])

Sin herramientas, el modelo **no puede consultar la base de conocimiento**: aquí el mock decide que necesita `search_kb`, así que `finish_reason="tool_calls"` y no hay texto. Eso es el germen del agente: la respuesta **no es el final**, es una *petición de acción*.

## 1.3 El bucle del agente, a mano (≈ 25 líneas)

Antes de usar `SupportAgent`, escribamos el bucle desde cero con un «modelo de juguete» (un guion) y una calculadora segura. Fíjate en la **lista de mensajes**: es la única memoria del agente.

In [ ]:
import ast, operator

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}
def calculadora(expr: str) -> float:
    """Evalúa aritmética básica SIN usar eval() (nunca evalúes texto que viene de un LLM)."""
    def ev(n):
        if isinstance(n, ast.Constant): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("expresión no permitida")
    return ev(ast.parse(expr, mode="eval").body)

def modelo_de_juguete(mensajes):
    """Sustituye al LLM: guion fijo «17*23 + 5». Devuelve ('tool', nombre, args) o ('final', texto)."""
    herramientas_usadas = [m for m in mensajes if m["role"] == "tool"]
    if len(herramientas_usadas) == 0: return ("tool", "calculadora", {"expr": "17*23"})
    if len(herramientas_usadas) == 1: return ("tool", "calculadora", {"expr": f"{herramientas_usadas[0]['content']}+5"})
    return ("final", f"El resultado es {herramientas_usadas[-1]['content']}.")

def agente_minimo(pregunta: str, max_pasos: int = 5):
    mensajes = [{"role": "user", "content": pregunta}]              # 1) memoria = lista de mensajes
    for paso in range(max_pasos):                                   # 2) condición de parada: máximo de iteraciones
        accion = modelo_de_juguete(mensajes)                        # 3) el «LLM» decide
        if accion[0] == "final":                                    # 4a) si no pide herramientas → terminamos
            return accion[1], mensajes
        _, nombre, args = accion                                    # 4b) si pide una herramienta → la ejecutamos
        resultado = calculadora(**args)
        mensajes.append({"role": "assistant", "content": f"{nombre}({args})"})
        mensajes.append({"role": "tool", "content": str(resultado)})   # 5) el resultado vuelve a la memoria
    return "No he podido terminar (máximo de pasos)", mensajes

respuesta, hist = agente_minimo("¿Cuánto es 17*23 + 5?")
for m in hist: print(f"{m['role']:>9}: {m['content']}")
print("\nRESPUESTA FINAL:", respuesta)

In [ ]:
fig, ax = viz.canvas(12, 4.2, (0, 100), (0, 38))
mem = viz.box(ax, 3, 14, 17, 10, "Mensajes\n(la memoria del\nagente)", viz.PALETTE["neutral"], bold=True)
llm_ = viz.box(ax, 29, 14, 14, 10, "LLM", viz.PALETTE["traces"], bold=True)
dec = viz.box(ax, 51, 14, 16, 10, "¿pide\nherramientas?", viz.PALETTE["warn"])
tool = viz.box(ax, 51, 1, 16, 8, "Ejecutar\nherramienta", viz.PALETTE["logs"], bold=True)
fin = viz.box(ax, 78, 14, 17, 10, "Respuesta\nfinal", viz.PALETTE["ok"], bold=True)
guard = viz.box(ax, 36, 28, 28, 7, "guarda: max_steps (condición de parada)", viz.PALETTE["error"], fontsize=8)
viz.arrow(ax, viz.right(mem), viz.left(llm_), "prompt")
viz.arrow(ax, viz.right(llm_), viz.left(dec), "respuesta"); viz.arrow(ax, viz.right(dec), viz.left(fin), "no")
viz.arrow(ax, viz.bottom(dec), viz.top(tool), "sí", viz.PALETTE["logs"])
viz.arrow(ax, (tool[0] - 0.3, tool[1] + 4), (mem[0] + 8.5, mem[1] - 0.4), "", viz.PALETTE["logs"], rad=-0.3)
ax.text(30, 4.2, "el resultado de la herramienta\nse añade a los mensajes", color=viz.PALETTE["logs"], fontsize=8, ha="center")
viz.arrow(ax, (guard[0] + 3, guard[1] - 0.3), (dec[0] + 3, dec[1] + 10.4), "", viz.PALETTE["error"], dashed=True)
plt.show()

> 👀 **Qué debes observar**
>
> - El agente es un `for` con una condición de parada: **no hay magia**, solo un LLM dentro de un bucle y una lista que crece.
> - Cada vuelta del ciclo *Mensajes → LLM → herramienta → Mensajes* es **un paso**. Cada paso = una llamada al modelo (latencia + tokens + dinero) y, a menudo, una llamada a un sistema externo (que puede fallar).
> - Sin la guarda `max_steps`, un modelo que nunca emite una respuesta final **gastaría tokens para siempre**. Esa guarda es tu primer control de coste (y su activación es una señal que querremos observar).

## 1.4 SupportOps v0: el mismo bucle, con proveedor, herramientas y reintentos

`SupportAgent` (en `supportops/agent.py`) es el mismo bucle más lo necesario para producción: proveedor LLM intercambiable, registro de herramientas con esquema, **reintentos** ante errores 5xx/429,
control de pasos, cálculo de tokens y coste, y *ganchos* para instrumentación (que de momento no hacen nada). Veamos el núcleo del código real:

In [ ]:
import inspect
src = inspect.getsource(SupportAgent.run).splitlines()
a = next(i for i, l in enumerate(src) if "for step in range" in l)
print("\n".join(src[a:a + 26]))
print("   ...")

Probemos las cuatro intenciones del negocio. Observa las columnas: **pasos**, **tokens** y **latencia simulada** de cada petición.

In [ ]:
agent = SupportAgent()
preguntas = [
    "¿Dónde está mi pedido ORD-00012?",
    "Quiero devolver el pedido ORD-00020 y que me devuelvan el dinero.",
    "¿Cuántos años de garantía tienen los productos?",
    "Quiero hablar con una persona real, esto es inaceptable.",
]
filas = []
for q in preguntas:
    r = agent.run(q)
    filas.append({"pregunta": q[:46] + "…", "estado": r.status, "herramientas": ",".join(r.tools_used), "pasos": len(r.steps),
                  "tokens_in": r.input_tokens, "tokens_out": r.output_tokens, "latencia_s": round(r.latency_s, 2), "respuesta": r.answer[:60] + "…"})
pd.DataFrame(filas)

In [ ]:
def explicar(r):
    """Cuenta la historia de una ejecución paso a paso (útil para depurar sin trazas… todavía)."""
    print(f"ticket {r.ticket_id}: estado={r.status} · {len(r.steps)} pasos · {r.latency_s:.2f}s simulados · {r.input_tokens}+{r.output_tokens} tokens · {r.cost_usd*1000:.3f} milésimas de USD")
    for s in r.steps:
        extra = f" tokens={s.input_tokens}/{s.output_tokens} finish={s.finish_reason}" if s.kind == "llm" else f" args={s.attributes.get('arguments')}"
        print(f"  {s.index}. {s.kind:4s} {s.name:18s} {s.latency_s:5.2f}s [{s.status}]{extra}")
    print("  →", r.answer)

r_ref = agent.run(Ticket("T-demo", "C-0001", "standard", "chat", "Quiero un reembolso del pedido ORD-00020, no me gustó.", "refund", SIM_EPOCH_S))
explicar(r_ref)
fig, ax = plt.subplots(figsize=(9, 3)); fig.subplots_adjust(left=0.22)
viz.timeline(r_ref.steps, ax, "Una petición de reembolso: 3 llamadas al LLM y 2 herramientas"); plt.show()

> 👀 **Qué debes observar**
>
> - El patrón **LLM → herramienta → LLM → herramienta → LLM** es la firma del agente: tres llamadas al modelo para dos herramientas.
> - El tiempo se reparte entre **espera del modelo** (barras azules, la mayor parte) y herramientas (verdes, mucho más cortas). Para optimizar latencia en agentes normalmente optimizas *llamadas al LLM* (menos pasos, prompts más cortos, modelos más rápidos), no tus APIs internas.
> - Esta gráfica la hemos dibujado nosotros con datos que el agente **devuelve**. En producción no tendrás un `AgentResult`: tendrás una **traza**, que es exactamente esta misma cascada pero capturada por instrumentación (módulo 03).

## 1.5 Teoría: mensajes, herramientas, parada y coste

**Mensajes.** Cada mensaje tiene un `role`: `system` (instrucciones permanentes), `user` (el cliente), `assistant` (el modelo; puede incluir `tool_calls`) y `tool` (el resultado de una herramienta, con el `tool_call_id` que lo enlaza).
El modelo **no tiene memoria entre llamadas**: en cada paso se le reenvía *toda* la conversación.

**Herramientas.** Se declaran con un nombre, una descripción y un **JSON Schema** de argumentos; el modelo devuelve `tool_calls` con el nombre y los argumentos, y **tu código** ejecuta la función. Las de SupportOps:

In [ ]:
import json
tools = default_tools()
print(json.dumps(tools.get("issue_refund").schema(), indent=1, ensure_ascii=False))
pd.DataFrame([{"herramienta": t, "descripción": tools.get(t).description[:70], "latencia mediana (s)": tools.get(t).latency_median_s} for t in tools.names()]).set_index("herramienta")

**Parada.** El bucle termina cuando el modelo responde **sin** `tool_calls` (`finish_reason="stop"`), cuando se agotan los pasos (`max_steps`) o ante un error no recuperable.

**Tokens y coste, primero con números.** Un agente de 3 llamadas. El prompt inicial (instrucciones + herramientas + pregunta) pesa 800 tokens. El modelo pide una herramienta (escribe 40 tokens) y esta devuelve 260.
En la 2.ª llamada se reenvía todo: 800 + 40 + 260 = **1100** tokens de entrada; si repite (+40 y +260) la 3.ª recibe **1400**. Entrada total = 800 + 1100 + 1400 = **3300**, no 3 × 800 = 2400. Eso es el efecto «el modelo no tiene memoria: se le reenvía toda la conversación». Ahora la fórmula general:

**Tokens y coste.** Sea $S$ el tamaño del contexto inicial (instrucciones del sistema + definición de herramientas + mensaje del cliente), $u_j$ los tokens de los resultados de herramienta añadidos tras el paso $j$ y $o_j$ los tokens que el modelo generó en ese paso.
En la llamada $k$ el modelo recibe:

$$ I_k \;=\; S \;+\; \sum_{j<k} (o_j + u_j) $$

y el coste de una petición de $K$ llamadas es

$$ \text{coste} \;=\; \frac{1}{10^6}\sum_{k=1}^{K} \big( I_k \cdot p_{in} \;+\; O_k \cdot p_{out} \big) $$

con $p_{in}, p_{out}$ los precios por millón de tokens de entrada y salida. Como $I_k$ crece con $k$, la **suma de tokens de entrada crece ~ cuadráticamente con el número de pasos**: duplicar los pasos puede *más que duplicar* el coste.

> 📐 Los precios del modo `mock` son **simulados** (`supportops/pricing.py`: 0,25 / 1,25 USD por millón para `mock-small`; 3 / 15 para `mock-large`). Para costes reales consulta la página de precios de tu proveedor.

In [ ]:
# Tokens de entrada por llamada al LLM: petición normal vs petición con bucle de herramientas (fallo inyectado)
base = Ticket("T-demo", "C-0001", "standard", "chat", "Quiero un reembolso del pedido ORD-00020, no me gustó.", "refund", SIM_EPOCH_S)
loop = Ticket("T-demo", "C-0001", "standard", "chat", base.text, "refund", SIM_EPOCH_S, faults=[FaultSpec("tool_loop")])
r_ok, r_loop = agent.run(base), agent.run(loop)
tok = lambda r: [s.input_tokens for s in r.steps if s.kind == "llm"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(np.arange(len(tok(r_ok))) - 0.2, tok(r_ok), 0.4, label=f"normal ({r_ok.n_llm_calls} llamadas)", color=viz.PALETTE["ok"])
ax[0].bar(np.arange(len(tok(r_loop))) + 0.2, tok(r_loop), 0.4, label=f"con bucle ({r_loop.n_llm_calls} llamadas)", color=viz.PALETTE["error"])
ax[0].set_xlabel("nº de llamada al LLM"); ax[0].set_ylabel("tokens de ENTRADA de esa llamada"); ax[0].set_title("Cada vuelta reenvía todo el contexto"); ax[0].legend()
ks = np.arange(1, 13); S, per = 350, 120
ax[1].plot(ks, S * ks + per * ks * (ks - 1) / 2, "o-", color=viz.PALETTE["metrics"]); ax[1].plot(ks, S * ks, "k:", label="si el contexto no creciera (lineal)")
ax[1].set_xlabel("nº de llamadas al LLM (K)"); ax[1].set_ylabel("tokens de entrada acumulados"); ax[1].set_title("Modelo: suma de I_k (S=350, +120 por paso)"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"normal: {r_ok.input_tokens} tokens in, {r_ok.cost_usd*1000:.3f} m$ · con bucle: {r_loop.input_tokens} tokens in, {r_loop.cost_usd*1000:.3f} m$ → {r_loop.cost_usd/r_ok.cost_usd:.1f}× el coste")

> 👀 **Qué debes observar**
>
> - A la izquierda, cada barra es **más alta que la anterior**: el agente reenvía el historial completo. El bucle (rojo) hace 6 llamadas y la última recibe ≈ 2,5× los tokens de la primera.
> - A la derecha, el modelo teórico: la curva naranja se separa de la línea recta, que sería el coste si cada llamada fuera independiente. Esto explica por qué **un bucle largo es tan caro** y por qué *recortar contexto* (resúmenes, no repetir herramientas) es una palanca de coste de primer orden (módulo 13).
> - El coste de la petición con bucle no es 2×: es varias veces el de la normal.

![El Bucle del Agente y el Coste Cuadrático del Contexto](assets/01_bucle_y_coste_cuadratico.png)
*Figura 1. Relación directa entre el bucle `Mensajes → LLM → Herramienta → Mensajes` y la acumulación cuadrática de tokens de entrada ($I_k$).* 

In [ ]:
# Determinismo: misma semilla → mismos resultados (por eso las rúbricas del curso son reproducibles)
t = generate_tickets(5, seed=SEED)[0]
a = SupportAgent(llm=MockLLM(seed=SEED)).run(t)
b = SupportAgent(llm=MockLLM(seed=SEED)).run(t)
c = SupportAgent(llm=MockLLM(seed=7)).run(t)
print("misma semilla  → latencia idéntica:", a.latency_s == b.latency_s, f"({a.latency_s:.4f} s)")
print("otra semilla   → latencia distinta:", c.latency_s != a.latency_s, f"({c.latency_s:.4f} s)")

> **🧠 Secretos de la élite · determinismo por llamada**
>
> El mock **no usa un generador aleatorio global**: cada llamada deriva su propio RNG de `SHA-256(seed, ticket_id, nº de llamada)`. Así la latencia de la llamada 3 del ticket T-17 es la misma
> se ejecute el notebook entero, solo esa celda o en otro orden. Un `random.seed(42)` global *no* da esa garantía (cualquier llamada extra desplaza toda la secuencia). Es el mismo principio que usan los sistemas de *record/replay* para depurar agentes reales.

---
# 2 · Proveedor LLM agnóstico

El agente no depende de ningún proveedor: habla con un `LLMClient` (un *Protocol*: cualquier objeto con `.chat(messages, tools, ...)` que devuelva `LLMResponse`). La fábrica `get_llm()` lee el entorno:

| `LLM_PROVIDER` | Qué usa | Variables |
|---|---|---|
| `mock` (por defecto) | `MockLLM`: determinista, sin red, sin coste | `LLM_MODEL=mock-small \| mock-large`, `SUPPORTOPS_SEED` |
| `anthropic` | SDK oficial `anthropic` (API Messages con `tools`) | `ANTHROPIC_API_KEY`; `LLM_MODEL` (por defecto `claude-sonnet-5-5`; verifica los modelos disponibles hoy) |
| `openai` | SDK `openai` (Chat Completions con `tools`) | `OPENAI_API_KEY`, `LLM_MODEL` **obligatoria**, opcional `OPENAI_BASE_URL` |
| `ollama` | API HTTP local `/api/chat` (sin dependencias) | `LLM_MODEL` **obligatoria** (un modelo con soporte de herramientas), `OLLAMA_HOST` |

> ⚠️ **Las API keys solo por variables de entorno** (`export ANTHROPIC_API_KEY=...` antes de lanzar Jupyter). Nunca en un notebook: acabaría en git. Las rúbricas del curso se calibran **solo con `mock`**.
> Los adaptadores reales están probados con clientes de prueba (sin red) y *no* contra las APIs en vivo durante la construcción del curso.

Cada adaptador traduce el formato interno a la API de su proveedor (por ejemplo, `tool_result` como bloque dentro de un mensaje `user` en Anthropic, o `role="tool"` en OpenAI) y **normaliza la respuesta** a `LLMResponse`. Mismo agente, distinto motor:

In [ ]:
from supportops.llm.anthropic_provider import to_anthropic_messages
from supportops.llm.openai_provider import to_openai_messages

conv = [Message("user", "¿Dónde está ORD-00012?"),
        Message("assistant", "", [ToolCall("call_1", "get_order", {"order_id": "ORD-00012"})]),
        Message("tool", '{"status": "processing"}', tool_call_id="call_1", name="get_order")]
print("— formato Anthropic —");  print(json.dumps(to_anthropic_messages(conv), indent=1, ensure_ascii=False)[:520], "...")
print("— formato OpenAI —");     print(json.dumps(to_openai_messages(conv, "sistema"), indent=1, ensure_ascii=False)[:420], "...")

In [ ]:
# Prueba OPT-IN con un proveedor real: solo si TÚ has exportado la clave y quieres gastar unos tokens.
provider = os.environ.get("LLM_PROVIDER", "mock")
if provider != "mock" and (os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("OPENAI_API_KEY") or provider == "ollama"):
    real = SupportAgent(llm=get_llm(provider))
    explicar(real.run("¿Cuántos años de garantía tienen los productos?"))
else:
    print("LLM_PROVIDER=mock → no se llama a ningún servicio externo. Para probar uno real:\n  export LLM_PROVIDER=anthropic ANTHROPIC_API_KEY=...  (antes de lanzar Jupyter)")

## 2.1 El modelo de latencia del mock

Para que las trazas y los dashboards se parezcan a los de producción, la latencia simulada de cada llamada es:

$$ \text{latencia} \;=\; \underbrace{T_{ttft}\, e^{\epsilon_1}}_{\text{tiempo hasta el primer token}} \;+\; \underbrace{o\cdot c_{out}\, e^{\epsilon_2}}_{\text{generación}} \;+\; \underbrace{I\cdot c_{in}}_{\text{prefill}} ,\qquad \epsilon_i\sim\mathcal N(0,\sigma_i^2) $$

con $o$ = tokens de salida, $I$ = tokens de entrada, y (para `mock-small`) $T_{ttft}=0{,}35$ s, $c_{out}=12$ ms/token, $c_{in}=0{,}04$ ms/token. Además, un 2 % de las llamadas son «lentas por naturaleza» (×3): la cola pesada típica.
Los ruidos son *lognormales* (multiplicativos), que es lo que se observa en latencias reales. Lo comprobamos con tráfico:

In [ ]:
N = 150 if FAST_DEV_RUN else 400
tk = generate_tickets(N, seed=SEED)                                   # tráfico SIN fallos: aislamos el modelo de latencia
res_s = run_tickets(SupportAgent(llm=MockLLM("mock-small", seed=SEED)), tk)
res_l = run_tickets(SupportAgent(llm=MockLLM("mock-large", seed=SEED)), tk)
llm_steps = lambda rs: pd.DataFrame([{"out": s.output_tokens, "inp": s.input_tokens, "lat": s.latency_s} for r in rs for s in r.steps if s.kind == "llm"])
Ds, Dl = llm_steps(res_s), llm_steps(res_l)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.scatter(Ds.out, Ds.lat, s=10, alpha=.45, label="mock-small", color=viz.PALETTE["traces"]); ax.scatter(Dl.out, Dl.lat, s=10, alpha=.45, label="mock-large", color=viz.PALETTE["metrics"])
xs = np.linspace(0, max(Ds.out.max(), Dl.out.max()), 50)
ax.plot(xs, 0.35 + 0.012 * xs + 0.00004 * Ds.inp.median(), color=viz.PALETTE["traces"], lw=2); ax.plot(xs, 0.70 + 0.030 * xs + 0.00008 * Dl.inp.median(), color=viz.PALETTE["metrics"], lw=2)
ax.set_xlabel("tokens de salida de la llamada"); ax.set_ylabel("latencia de la llamada al LLM (s, simulada)"); ax.set_title("Latencia ≈ TTFT + tokens·s/token (+ cola lenta natural)"); ax.legend()
plt.show()
print(f"mock-small: mediana {Ds.lat.median():.2f}s · p99 {Ds.lat.quantile(.99):.2f}s | mock-large: mediana {Dl.lat.median():.2f}s · p99 {Dl.lat.quantile(.99):.2f}s")

> 👀 **Qué debes observar**
>
> - Los puntos siguen una **recta** (TTFT + pendiente por token). `mock-large` tiene mayor ordenada (TTFT) y mayor pendiente: es más lento en ambos conceptos.
> - Los puntos muy por encima de la recta son la **cola lenta natural** (×3). No son un fallo inyectado: así es la latencia de un proveedor real. Un SLO de latencia debe vivir con esa cola.
> - Observa también que las llamadas con **pocos tokens de salida** (las que piden herramienta) son rápidas: el tiempo dominante es el TTFT, no la generación.

In [ ]:
# Mismo tráfico, dos modelos: ¿qué compras con un modelo «grande»? (en el mock: más latencia y 12× el precio)
def resumen(rs):
    d = pd.DataFrame([r.to_dict() for r in rs])
    return {"latencia media (s)": d.latency_s.mean(), "p95 latencia (s)": d.latency_s.quantile(.95), "coste medio / ticket (m$)": d.cost_usd.mean() * 1000, "tokens in / ticket": d.input_tokens.mean()}
cmp = pd.DataFrame({"mock-small": resumen(res_s), "mock-large": resumen(res_l)}).T
display(cmp.round(2))
fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
for a_, col, lab in zip(ax, ["latencia media (s)", "p95 latencia (s)", "coste medio / ticket (m$)"], ["segundos", "segundos", "milésimas de USD"]):
    a_.bar(cmp.index, cmp[col], color=[viz.PALETTE["traces"], viz.PALETTE["metrics"]]); a_.set_title(col); a_.set_ylabel(lab)
plt.tight_layout(); plt.show()
print(f"coste ×{cmp.loc['mock-large', 'coste medio / ticket (m$)'] / cmp.loc['mock-small', 'coste medio / ticket (m$)']:.1f} · latencia ×{cmp.loc['mock-large', 'latencia media (s)'] / cmp.loc['mock-small', 'latencia media (s)']:.1f}")

> 👀 **Qué debes observar**
>
> - El coste por ticket se multiplica por el cociente de precios (12×) mientras la latencia «solo» se duplica aproximadamente. En la práctica **la decisión de modelo es una decisión de coste y de latencia a la vez**, y tiene que verse en los dashboards (módulos 09 y 13).
> - Las gráficas son **comparables** porque el tráfico es idéntico (misma semilla): ése es el valor de un generador reproducible.

---
# 3 · Herramientas y base de conocimiento (RAG)

## 3.1 Las herramientas simuladas

| Herramienta | Qué hace | Regla de negocio / fallo posible |
|---|---|---|
| `get_order(order_id)` | Estado de un pedido (de 600 sintéticos `ORD-00001…`) | `found: false` si no existe |
| `issue_refund(order_id, amount_eur, reason)` | Reembolsa | Solo pedidos **entregados hace ≤ 30 días** y por importe ≤ total. Si no: `approved: false` con `reason` |
| `search_kb(query, k)` | Busca en 14 documentos (devoluciones, garantía, envíos, drivers…) | Devuelve `score` por documento |
| `escalate_to_human(reason, priority)` | Deriva a una persona | — |

**Error de negocio ≠ error de infraestructura.** Que `issue_refund` devuelva `approved: false` es un **resultado normal** (el cliente no cumple la política) y *no* debe contar como error del sistema.
Una excepción `ToolError` (timeout, 5xx del servicio de pedidos) sí lo es. Confundirlos infla la tasa de error y despierta a on-call por nada — o peor, oculta fallos reales.

In [ ]:
from supportops.tools import issue_refund, get_order, ToolError
from supportops import catalog
orders = catalog.orders()
elegible = next(o for o in orders.values() if o["status"] == "delivered" and o["delivered_days_ago"] <= 30)
antiguo = next(o for o in orders.values() if o["status"] == "delivered" and o["delivered_days_ago"] > 30)
enviado = next(o for o in orders.values() if o["status"] == "shipped")
for o in (elegible, antiguo, enviado):
    print(f"{o['order_id']} ({o['status']}, entregado hace {o['delivered_days_ago']} d) →", issue_refund(o["order_id"], o["total_eur"]))
print("\nError de infraestructura (distinto):")
try:
    raise ToolError("el servicio de pedidos devolvió 503", "upstream_503")
except ToolError as e:
    print(" ToolError:", e, "| error_type =", e.error_type)

## 3.2 RAG sencillo: TF-IDF + coseno

**RAG** (*retrieval-augmented generation*, Lewis et al., 2020) = antes de responder, **recuperar** documentos relevantes y dárselos al modelo. Aquí la recuperación es TF-IDF, sin dependencias:

$$ \text{idf}(t)=\ln\frac{1+n}{1+\text{df}(t)}+1,\qquad \vec d_t=\text{tf}(t,d)\cdot\text{idf}(t),\qquad \text{score}(q,d)=\cos(\vec q,\vec d)=\frac{\vec q\cdot\vec d}{\lVert\vec q\rVert\,\lVert\vec d\rVert} $$

con $n$ documentos y $\text{df}(t)$ cuántos contienen el término $t$. Un término raro (alto idf) pesa más que uno frecuente. Veamos qué recupera para tres consultas, incluida una que **no encaja bien**:

In [ ]:
from supportops.kb import KnowledgeBase
kb = KnowledgeBase()
consultas = ["¿cuántos años de garantía tienen?", "mi tarjeta gráfica parpadea tras instalarla", "¿cuánto tarda el correo en llegar a casa?"]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharex=True)
for ax, q in zip(axes, consultas):
    res = kb.search(q, k=4)
    ax.barh([f"{r['doc_id']}\n{r['title'][:24]}" for r in res][::-1], [r["score"] for r in res][::-1], color=viz.PALETTE["logs"] if res and res[0]["score"] > 0.2 else viz.PALETTE["warn"])
    ax.set_title(q if len(q) < 40 else q[:38] + "…", fontsize=9); ax.set_xlabel("similitud coseno")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - Con vocabulario compartido («garantía», «tarjeta gráfica», «parpadea») el documento correcto sale **primero con una puntuación clara**.
> - La tercera consulta usa sinónimos («correo», «a casa») que el TF-IDF no entiende: las puntuaciones son bajas y **parecidas entre sí** (barras amarillas). Si el agente aun así responde con el primer documento, **inventa con apariencia de fundamento**. Una puntuación de recuperación baja es una *señal de calidad* que merece ser un atributo de la traza (módulo 04) y una métrica (módulo 14).
> - Los embeddings densos resuelven sinónimos, pero introducen otro coste (cómputo, índice, deriva) — y otros fallos que observar.

---
# 4 · Tráfico sintético con semilla

## 4.1 El generador

`generate_tickets(n, seed, duration_s, fault_rates, incidents, start_s, ...)` crea **tickets** de clientes ficticios. Es determinista: misma semilla → mismos tickets. Esquema:

In [ ]:
print(schema.describe(schema.TICKET_SCHEMA))
tk = generate_tickets(400 if not FAST_DEV_RUN else 200, seed=SEED, fault_rates=DEFAULT_FAULT_RATES,
                      incidents=[Incident("llm_provider_degraded", "llm_error", 1200, 1800, prob=0.7, params={"status": 503, "persistent": True})])
print("\nejemplo:", tk[0].to_dict()["text"], "| intent:", tk[0].intent_true, "| faults:", [f.kind for f in tk[0].faults])

In [ ]:
T = pd.DataFrame([t.to_dict() | {"minute": (t.created_at - SIM_EPOCH_S) / 60, "n_faults": len(t.faults)} for t in tk])
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].hist(T.minute, bins=np.arange(0, 61, 2), color=viz.PALETTE["traces"]); ax[0].axvspan(20, 30, color=viz.PALETTE["error"], alpha=.15, label="incidente (min 20–30)")
ax[0].set_xlabel("minuto"); ax[0].set_ylabel("tickets cada 2 min"); ax[0].set_title("Llegadas: pico a mitad de ventana"); ax[0].legend()
mix = T.intent_true.value_counts(normalize=True); ax[1].bar(mix.index, mix.values * 100, color=viz.PALETTE["logs"]); ax[1].set_ylabel("% de tickets"); ax[1].set_title("Mezcla de intenciones"); ax[1].tick_params(axis="x", rotation=20)
cnt = T.customer_id.value_counts().values; ax[2].loglog(np.arange(1, len(cnt) + 1), cnt, "o", color=viz.PALETTE["metrics"], ms=4); ax[2].set_xlabel("ranking del cliente (log)"); ax[2].set_ylabel("nº de tickets (log)"); ax[2].set_title("Cola larga de clientes (Zipf)")
plt.tight_layout(); plt.show()
print(f"{(T.customer_id.value_counts().head(10).sum() / len(T)):.0%} de los tickets vienen de los 10 clientes más activos de {T.customer_id.nunique()} distintos")

> 👀 **Qué debes observar**
>
> - **Izquierda:** llegadas con un pico central (patrón diario comprimido a 1 hora). La franja roja es una **ventana de incidente** que programamos: dentro de ella los tickets tienen un 70 % de probabilidad de sufrir un fallo del proveedor LLM.
> - **Centro:** la mezcla pedida (35 / 20 / 35 / 10 %).
> - **Derecha:** distribución de cola larga: unos pocos clientes generan muchísimo tráfico. Esto importa para observabilidad: **atribuir coste o errores por cliente** (alta cardinalidad) es útil en trazas y logs, pero peligroso como *label* de métricas (módulo 02 y 09).
> - Para enviar tráfico a un stack **vivo** usa `start_s="live"`: así los timestamps caen en la última hora y Tempo/Loki los aceptan.

---
# 5 · Fallos inyectados: el *ground truth* de la observabilidad

Siete tipos de fallo, activables **por ticket** (van dentro de `ticket.faults`). Como los inyectamos nosotros, sabemos la verdad y podremos medir si nuestra observabilidad los detecta.

In [ ]:
pd.DataFrame([{"fallo": k, "qué hace": v} for k, v in FAULT_KINDS.items()]).set_index("fallo")

Ejecutemos **el mismo ticket** (un reembolso) sin fallo y con cada fallo. Compara la fila «ninguno» con el resto:

In [ ]:
base = Ticket("T-demo", "C-0001", "standard", "chat", "Quiero un reembolso del pedido ORD-00020, no me gustó.", "refund", SIM_EPOCH_S)
casos = [("ninguno", None, {}), ("latency_spike ×8", "latency_spike", {"factor": 8.0, "target": "both"}), ("llm_error (transitorio)", "llm_error", {"persistent": False}),
         ("llm_error (persistente)", "llm_error", {"persistent": True}), ("tool_error", "tool_error", {}), ("tool_timeout", "tool_timeout", {}),
         ("hallucination", "hallucination", {}), ("tool_loop", "tool_loop", {}), ("pii_leak", "pii_leak", {"victim": "C-0042"})]
filas = []
for nombre, kind, params in casos:
    t = Ticket(base.ticket_id, base.customer_id, base.tier, base.channel, base.text, base.intent_true, base.created_at, faults=[FaultSpec(kind, params)] if kind else [])
    r = SupportAgent().run(t)
    filas.append({"caso": nombre, "estado": r.status, "latencia_s": round(r.latency_s, 2), "pasos": len(r.steps), "tokens_in": r.input_tokens,
                  "¿fundamentada?": r.grounded, "¿PII ajena?": r.pii_leaked, "error_type": r.error_type or "", "respuesta": r.answer[:55] + "…"})
F = pd.DataFrame(filas).set_index("caso")
F

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.9), sharey=True)
y = np.arange(len(F))
col = lambda v: viz.PALETTE["error"] if v in ("error", "max_steps") else viz.PALETTE["warn"] if v == "degraded" else viz.PALETTE["ok"]
ax[0].barh(y, F.latencia_s, color=[col(s) for s in F.estado]); ax[0].set_yticks(y); ax[0].set_yticklabels(F.index); ax[0].invert_yaxis(); ax[0].set_xlabel("latencia (s)"); ax[0].set_title("Latencia (color = estado)")
ax[1].barh(y, F.tokens_in, color=[col(s) for s in F.estado]); ax[1].set_xlabel("tokens de entrada"); ax[1].set_title("Tokens de entrada")
ax[2].barh(y, F.pasos, color=[col(s) for s in F.estado]); ax[2].set_xlabel("pasos"); ax[2].set_title("Pasos del agente")
for i, (g, p) in enumerate(zip(F["¿fundamentada?"], F["¿PII ajena?"])):
    if not g or p: ax[2].text(F.pasos.iloc[i] + 0.2, i, "respuesta MALA con estado «ok»" if F.estado.iloc[i] == "ok" else "", va="center", fontsize=8, color=viz.PALETTE["error"])
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - **`hallucination` es el caso más barato y rápido** (0,6 s, 1 paso, ≈ 350 tokens) frente al reembolso correcto (≈ 2,5 s, 5 pasos, ≈ 1300 tokens). Una «mejora de latencia y coste» puede ser un **síntoma de que el agente dejó de usar sus herramientas**. Un dashboard que celebre ese descenso estaría celebrando una regresión.
> - **`pii_leak` es idéntico al caso sano** en latencia, pasos y tokens (la fuga añade unos pocos tokens a la respuesta): ninguna métrica de infraestructura lo ve. Solo se detecta **mirando el contenido** (módulo 20).
> - **`tool_loop`** dispara pasos y tokens (≈ 2,8× los de entrada) y termina en `max_steps`: sí se ve en métricas de coste/pasos y, mejor aún, en la traza (la misma herramienta repetida).
> - **`tool_timeout` y `tool_error`** terminan en estado `degraded` (el agente se disculpa pero *responde*): el error está **dentro** de la traza, no en el estado de la petición raíz. Mira el span de la herramienta, no solo el raíz.
> - **`llm_error` transitorio** *no* acaba en error: el reintento lo absorbe y solo se nota como +1 llamada y más latencia. **Persistente** sí falla (estado `error`). Los reintentos *enmascaran* problemas: hay que contarlos aparte.

![Huella Telemétrica de los 7 Fallos en SupportOps](assets/02_huellas_siete_fallos.png)
*Figura 2. Guía de diagnóstico rápido: qué firma deja cada fallo inyectado en estado raíz, latencia, pasos, coste y spans hijos.*

In [ ]:
# ¿Cómo se ve cada fallo en la línea de tiempo? (latencia de los pasos)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4)); fig.subplots_adjust(left=0.08, wspace=0.55)
for ax, (nombre, kind, params) in zip(axes, [casos[0], casos[5], casos[7]]):
    t = Ticket(base.ticket_id, base.customer_id, base.tier, base.channel, base.text, base.intent_true, base.created_at, faults=[FaultSpec(kind, params)] if kind else [])
    r = SupportAgent().run(t); viz.timeline(r.steps[:12], ax, f"{nombre}  ({r.status})")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> Tres «huellas» distintas en la línea de tiempo: la **sana** (5 pasos cortos), el **timeout** (una barra roja de 5 s en la herramienta y un LLM que responde con disculpa) y el **bucle** (la misma herramienta repetida 6 veces, cada LLM más lenta por el contexto acumulado).
> Este tipo de vistas es lo que vas a obtener automáticamente de Tempo; hoy la dibujamos a mano para saber qué buscar.

---
# 6 · Experimento: tráfico sano vs tráfico con fallos de fondo

Ejecutamos el mismo tráfico dos veces: sin fallos y con las tasas por defecto (`DEFAULT_FAULT_RATES`). ¿Qué cambia en las métricas agregadas que un dashboard clásico mostraría?

In [ ]:
N = 300 if FAST_DEV_RUN else 1000
tk_ok = generate_tickets(N, seed=SEED)
tk_bad = generate_tickets(N, seed=SEED, fault_rates=DEFAULT_FAULT_RATES)
d_ok = pd.DataFrame(to_rows(tk_ok, run_tickets(SupportAgent(), tk_ok)))
d_bad = pd.DataFrame(to_rows(tk_bad, run_tickets(SupportAgent(), tk_bad)))
def kpis(d):
    bad = (~d.grounded) | d.pii_leaked
    return {"error rate (error+max_steps)": d.status.isin(["error", "max_steps"]).mean() * 100, "degraded %": d.status.eq("degraded").mean() * 100,
            "respuestas malas con estado ok %": (bad & d.status.eq("ok")).mean() * 100, "p50 latencia (s)": d.latency_s.quantile(.5), "p99 latencia (s)": d.latency_s.quantile(.99),
            "coste total (m$)": d.cost_usd.sum() * 1000, "tokens in / ticket": d.input_tokens.mean()}
K = pd.DataFrame({"sano": kpis(d_ok), "con fallos": kpis(d_bad)}); K["Δ"] = K["con fallos"] - K["sano"]
display(K.round(2))
fig, ax = plt.subplots(1, 3, figsize=(14, 3.7))
for d, lab, c in [(d_ok, "sano", viz.PALETTE["ok"]), (d_bad, "con fallos", viz.PALETTE["error"])]:
    l = np.sort(d.latency_s.values); ax[0].plot(l, np.arange(1, len(l) + 1) / len(l), label=lab, color=c, lw=2)
ax[0].set_xscale("log"); ax[0].set_xlabel("latencia (s)"); ax[0].set_ylabel("ECDF"); ax[0].set_title("Latencia: la cola engorda"); ax[0].legend()
st = pd.DataFrame({"sano": d_ok.status.value_counts(normalize=True), "con fallos": d_bad.status.value_counts(normalize=True)}).fillna(0) * 100
SC = {"ok": viz.PALETTE["ok"], "escalated": viz.PALETTE["traces"], "degraded": viz.PALETTE["warn"], "error": viz.PALETTE["error"], "max_steps": viz.PALETTE["dark"]}
st.T.plot(kind="bar", stacked=True, ax=ax[1], color=[SC[c] for c in st.index], legend=True); ax[1].set_ylabel("% de tickets"); ax[1].set_title("Estados"); ax[1].tick_params(axis="x", rotation=0)
ax[2].boxplot([d_ok.cost_usd * 1000, d_bad.cost_usd * 1000], tick_labels=["sano", "con fallos"], flierprops=dict(marker=".", alpha=.5)); ax[2].set_ylabel("coste por ticket (m$)"); ax[2].set_title("Coste por ticket")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - La mediana casi no se mueve (los fallos son **minoría**), pero **p99 y máximo** sí: la cola es donde viven los problemas. Un SLO sobre la mediana no detectaría nada.
> - En «Estados», la barra de «con fallos» muestra errores y `degraded`, pero **la fila «respuestas malas con estado ok» de la tabla** es la que no aparece en ningún gráfico de estados: un SLI de éxito basado en estado la ignora.
> - El **coste total no sube; incluso baja un poco**: las alucinaciones (que no llaman herramientas) y las peticiones fallidas son *baratas*. Un dashboard de gasto vería un «ahorro» justo cuando la calidad empeora. Mientras tanto, el `coste por ticket` muestra extremos (bucles): el gasto está **concentrado**, como vimos en el módulo 00.

---
# 7 · Puntos de extensión: dónde se instrumentará (adelanto del módulo 03)

El agente v0 **no está instrumentado**, pero llama a un `Instrumentor` (por defecto no hace nada) en tres sitios, con *context managers*:

```python
with instr.agent_run(ctx):                  # 1 por ticket          → span raíz   «invoke_agent supportops-agent»
    with instr.llm_call(ctx, info): ...     # 1 por llamada al LLM → span        «chat <modelo>»
    with instr.tool_call(ctx, info): ...    # 1 por herramienta    → span        «execute_tool <herramienta>»
```

Para enseñarte el **modelo de datos** sin depender aún de OpenTelemetry, el paquete trae `RecordingInstrumentor`: guarda *spans* y *logs* en memoria con identificadores, padre/hijo, atributos y estado.
No es OpenTelemetry (eso es el módulo 03), pero tiene su misma forma. Veamos qué captura de una ejecución:

In [ ]:
rec = RecordingInstrumentor()
r = SupportAgent(instrumentor=rec).run(Ticket("T-demo", "C-0001", "premium", "chat", "Quiero un reembolso del pedido ORD-00020, no me gustó.", "refund", SIM_EPOCH_S,
                                            faults=[FaultSpec("tool_timeout")]))
sp = pd.DataFrame(rec.spans)
sp["dur_s"] = (sp.end_unix_nano - sp.start_unix_nano) / 1e9
sp["padre"] = sp.parent_span_id.fillna("—").str[:6]
display(sp[["span_id", "padre", "name", "kind", "status", "dur_s"]].assign(span_id=sp.span_id.str[:6]).round(2))
print("atributos del span raíz:", {k: v for k, v in rec.spans[0]["attributes"].items() if k.startswith(("gen_ai", "supportops"))})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 3.6), gridspec_kw={"width_ratios": [1.5, 1]})
fig.subplots_adjust(left=0.26, wspace=0.08)
viz.waterfall(rec.spans, ax=ax[0], title="Cascada de la traza (spans registrados)")
lg = pd.DataFrame(rec.logs); lg["t_s"] = (lg.time_unix_nano - rec.spans[0]["start_unix_nano"]) / 1e9
ax[1].axis("off"); ax[1].set_title("Logs correlacionados (mismo trace_id)")
txt = "\n".join(f"{x.t_s:5.2f}s  {x.severity_text:5s} {x.body}   [span {x.span_id[:4]}]" for x in lg.itertuples())
ax[1].text(0, 0.95, txt, va="top", family="monospace", fontsize=8)
plt.show()
print("¿todos los logs comparten el trace_id de la traza?", set(lg.trace_id) == {rec.spans[0]["trace_id"]})

> 👀 **Qué debes observar**
>
> - La cascada muestra el **árbol**: raíz `invoke_agent`, y debajo `chat` y `execute_tool` (en rojo, el timeout de 5 s). El padre se identifica por `parent_span_id`.
> - Los **logs** del agente llevan el mismo `trace_id` y el `span_id` del span activo: así, desde una traza lenta saltarás a *sus* logs y viceversa. Esa correlación —que no es magia, es copiar dos identificadores— es la base del módulo 08.
> - Los nombres `invoke_agent`, `chat` y `execute_tool` y los atributos `gen_ai.*` siguen las **GenAI semantic conventions** de OpenTelemetry, que están en estado **Development** (inestables); las estudiaremos y validaremos en el módulo 04.

---
# 8 · 🏭 En producción

- **Anthropic** ([«Building effective agents»](https://www.anthropic.com/engineering/building-effective-agents), 2024) recomienda empezar por la solución más simple, observa que los agentes **intercambian latencia y coste por rendimiento**, que su autonomía implica **más coste y errores acumulados**, y que conviene incluir **condiciones de parada** (máximo de iteraciones) y pruebas extensas en entornos aislados. Cada una de esas recomendaciones se traduce en algo que observar: pasos por petición, coste por petición, activaciones de `max_steps`.
- **ReAct** (Yao et al., 2022) popularizó el patrón de intercalar *razonamiento y acciones* con herramientas, base de casi todos los agentes actuales; **RAG** (Lewis et al., 2020) el de recuperar documentos antes de generar. Ambos son fuentes de latencia (más pasos) y de fallos nuevos (recuperación irrelevante).
- **Documentación de proveedores:** el bucle `tool_use`/`tool_result` de Anthropic y las `tools` de OpenAI son el contrato que implementan nuestros adaptadores; ambos exigen que **tu código** ejecute la herramienta y devuelva el resultado en el siguiente mensaje (por eso el coste de contexto crece con cada paso).
- **Honeycomb** insiste (en sus artículos sobre su *Query Assistant*) en medir el éxito de las funciones con LLM con datos de uso reales y no solo con pruebas previas; es el motivo de que SupportOps nazca con *ground truth* de fallos para validar la observabilidad.

> **🧠 Secretos de la élite · agentes en producción**
>
> 1. **La unidad de coste y de fallo es el *paso*, no la petición.** Dos peticiones idénticas pueden costar 1× y 6×. Todo presupuesto razonable tiene `max_steps` *y* un tope de tokens/coste por petición (módulo 13).
> 2. **Los reintentos enmascaran fallos.** Un `llm_error` transitorio acaba en estado `ok`; si solo mides errores finales, no verás que el proveedor se degrada hasta que los reintentos se agotan. Cuenta **intentos**, no solo resultados.
> 3. **Herramientas con efectos (reembolsos, escrituras) deben ser idempotentes.** Un reintento tras un timeout puede ejecutar la acción dos veces. Usa claves de idempotencia; es un riesgo que ninguna métrica te mostrará hasta que haya un cliente reembolsado dos veces.
> 4. **Distingue error de negocio de error de infraestructura** (`approved:false` vs `ToolError`). Los SLOs de fiabilidad se calculan sobre lo segundo.
> 5. **«Más rápido y más barato» puede ser peor**: una alucinación no llama herramientas. Vigila `tool_calls por petición` como señal de salud, no solo latencia y coste.
> 6. **El contexto crece cuadráticamente en coste**: resumir o podar el historial suele ahorrar más que cambiar de modelo.
> 7. **Determinismo reproducible = capacidad de depuración.** Si puedes re-ejecutar un incidente con la misma semilla/entrada, conviertes un fallo «no determinista» en un *test*. En producción real, graba entradas y respuestas de herramientas para *replay*.
> 8. **`temperature=0` no garantiza determinismo** en proveedores reales (paralelismo, versiones del modelo, enrutado): no diseñes tests que lo asuman. *(Fuente secundaria / no verificada contra la página oficial de la API: varias guías de terceros lo indican; la documentación de cada proveedor lo matiza, compruébalo en la tuya.)*
> 9. **Mide el coste cuadrático, no lo supongas.** Si cada paso reenvía el historial completo, los tokens de *entrada* acumulados de una petición de $n$ pasos son $n\,c_0 + d\,n(n-1)/2$ ($c_0$ = prompt fijo, $d$ = tokens que añade cada paso): con los valores de la celda, pasar de 3 a 6 pasos multiplica el coste de entrada por ≈ 2,9, **no** por 2. La celda siguiente lo calcula.
> 10. **La caché de prompt es la palanca del coste cuadrático**, pero cambia lo que ves: en las APIs con caché explícita los tokens leídos de caché se facturan a una fracción del precio de entrada (cifras habituales: lectura ≈ 0,1× y escritura ≈ 1,25× el precio base en Anthropic; *fuente secundaria / no verificada: consulta la página de precios vigente*). Por eso el módulo 04 separa los tokens de caché en atributos propios: sin ellos, tu coste calculado no cuadra con la factura.

In [ ]:
# Coste de entrada acumulado de una petición con n pasos, reenviando todo el historial en cada paso
C0, D = 600, 250        # tokens del prompt fijo y tokens que añade cada paso (hipótesis ilustrativas)
def input_tokens(n: int) -> int:
    return sum(C0 + D * k for k in range(n))      # paso k reenvía C0 + k*D
for n in (1, 3, 6, 12):
    print(f"n={n:2d} pasos → {input_tokens(n):6d} tokens de entrada  ({input_tokens(n) / input_tokens(1):4.1f}× el de 1 paso)")

> **⚠️ Cuidado · Errores comunes (pitfalls)**
>
> - **Escribir tests de «la respuesta debe ser exactamente X»** contra un LLM real: usa propiedades (llamó a la herramienta correcta, citó el documento, no incluyó PII).
> - **Medir solo el estado final**: un agente puede responder `ok` tras tres herramientas fallidas.
> - **Olvidar `max_steps`** (o ponerlo muy alto): el primer bucle te cuesta dinero real.
> - **Evaluar con `eval()`** el texto que genera un LLM: ejecución de código arbitrario. Usa parsers o herramientas con argumentos tipados.
> - **Mezclar API keys en notebooks o repos**: siempre variables de entorno / gestor de secretos.
> - **Usar un RNG global** para simular: los resultados dependen del orden de ejecución y no son reproducibles.
> - **Suponer que el mock = realidad**: es un simulador con latencias y tokens plausibles, útil para aprender y para pruebas deterministas, no un sustituto de medir tu proveedor real.

---
# 9 · Autoevaluación

**Pregunta 1 · (Cálculo) Coste de una petición**

Una petición hace 3 llamadas al LLM con 800, 1100 y 1400 tokens de entrada y 40, 40 y 90 de salida. Con 0,25 USD/M de entrada y 1,25 USD/M de salida, ¿cuál es el coste? ¿Y si el agente hiciera 6 llamadas añadiendo ≈ 300 tokens de entrada por paso desde 800?

<details>
<summary>Ver respuesta</summary>

3 llamadas: entrada 800+1100+1400 = 3300 → 3300·0,25/10⁶ = 0,000825 USD; salida 170·1,25/10⁶ = 0,0002125 USD; **total ≈ 0,00104 USD (≈ 1,04 milésimas)**.
6 llamadas: entrada 800, 1100, 1400, 1700, 2000, 2300 = 9300 tokens → 0,0023 USD solo de entrada: **más del doble** con el doble de llamadas, por el crecimiento del contexto (efecto cuadrático).

</details>

**Pregunta 2 · (Cálculo) Latencia esperada**

Con `mock-small` (TTFT 0,35 s; 12 ms/token de salida; 0,04 ms/token de entrada), ¿qué latencia *media* (sin ruido) tiene una llamada con 900 tokens de entrada y 60 de salida? ¿Y una que solo pide una herramienta (≈ 25 tokens de salida)?

<details>
<summary>Ver respuesta</summary>

Con 60 de salida: 0,35 + 60·0,012 + 900·0,00004 = 0,35 + 0,72 + 0,036 = **≈ 1,11 s**. Con 25 de salida: 0,35 + 0,30 + 0,036 = **≈ 0,69 s**. Dominan TTFT y generación; el prefill apenas cuenta aquí.

</details>

**Pregunta 3 · (Diagnóstico) «El agente va mejor»**

Tras un cambio de prompt, la latencia media baja un 40 %, el coste un 60 % y el error rate no cambia. ¿Celebras? ¿Qué miras antes?

<details>
<summary>Ver respuesta</summary>

No celebres todavía. Una caída conjunta de latencia y coste con error rate estable es la huella de **menos pasos/herramientas**: quizá el agente ahora responde sin consultar (alucinaciones). Mira `tool_calls` por petición, la tasa de respuestas *fundamentadas* (evaluación online) y una muestra de trazas antes/después.

</details>

**Pregunta 4 · (Diagnóstico) Reembolsos duplicados**

Aparecen reembolsos duplicados en contabilidad, pero tus dashboards muestran 0 errores. Da una hipótesis basada en lo visto y cómo la confirmas con telemetría.

<details>
<summary>Ver respuesta</summary>

Hipótesis: un **timeout** en `issue_refund` seguido de un reintento (del LLM o de la herramienta) ejecutó la acción dos veces; la herramienta no es idempotente. Confirmación: en las trazas, peticiones con **dos spans `execute_tool issue_refund`** (el primero con error/timeout) para el mismo `order_id`; en logs, mismo `order_id` aprobado dos veces.

</details>

**Pregunta 5 · (Diagnóstico) Semillas**

Corres 1000 tickets con `seed=42` y obtienes p99 = 12,0 s. Un compañero obtiene 12,3 s con el mismo código. ¿Qué dos explicaciones razonables hay?

<details>
<summary>Ver respuesta</summary>

(1) **Distinto `SUPPORTOPS_SEED` o versión del paquete/mock** (el RNG deriva de la semilla y del código). (2) Usó otro `LLM_MODEL`/otras `fault_rates`. Si todo coincide, el resultado **debe** ser idéntico (el RNG es por llamada): una diferencia indicaría estado oculto o dependencia del orden — un bug.

</details>

**Pregunta 6 · (Transferencia) De tu bot actual**

Piensa en un chatbot o asistente que conozcas. ¿Es workflow o agente según la definición de Anthropic? ¿Qué *condición de parada* tiene y qué métrica te avisaría de que falla?

<details>
<summary>Ver respuesta</summary>

Si el orden de pasos está fijado por código (p. ej. «clasifica → busca → responde») es un *workflow*; si el modelo decide qué herramienta usar y cuándo parar, es un *agente*. La condición de parada típica es `max_steps`/tiempo máximo; la métrica: tasa de peticiones que terminan por esa condición (en SupportOps, estado `max_steps`).

</details>

**Pregunta 7 · (Transferencia) RAG en tu dominio**

Tu empresa tiene un RAG sobre documentación interna. ¿Qué dos señales de la recuperación incluirías como atributos de traza para detectar respuestas «con apariencia de fundamento»?

<details>
<summary>Ver respuesta</summary>

La **puntuación del mejor documento** (y la diferencia con el segundo) y el **número de documentos recuperados por encima de un umbral**; opcionalmente los *ids* de documentos. Una puntuación baja o plana indica recuperación débil: la respuesta probablemente no esté fundamentada.

</details>

**Pregunta 8 · (Transferencia) ¿Dónde mides los reintentos?**

Quieres medir cuánto tiempo pierde el agente en reintentos al LLM (errores 5xx/429). De los tres puntos de extensión (`agent_run`, `llm_call`, `tool_call`), ¿cuál usarías y qué registrarías para distinguir una llamada normal de una reintentada?

<details>
<summary>Ver respuesta</summary>

`llm_call`: es un span por cada llamada al modelo, así que un reintento aparece como **otro** span hijo del mismo `agent_run`. Registrarías un atributo con el número de intento (y el código de error del intento fallido) para poder contar intentos, no solo resultados finales. En `agent_run` solo verías el total, y en `tool_call` no ocurre el reintento del LLM.

</details>

**Pregunta 9 · (Cálculo) Cambiar de proveedor**

Quieres ejecutar el mismo agente contra un modelo local con Ollama. ¿Qué variables de entorno cambias y qué parte del código del agente tienes que tocar? ¿Por qué las rúbricas del curso siguen usando solo `mock`?

<details>
<summary>Ver respuesta</summary>

Cambias `LLM_PROVIDER=ollama`, `LLM_MODEL=<un modelo con soporte de herramientas>` y, si no es local por defecto, `OLLAMA_HOST`. **No tocas el agente**: habla con el protocolo `LLMClient` y `get_llm()` elige el adaptador. Las rúbricas usan `mock` porque es determinista, gratis y sin red: con un modelo real las cifras variarían entre ejecuciones y no se podrían fijar umbrales.

</details>

**Pregunta 10 · (Recuerdo) Roles y parada**

¿Qué roles tienen los mensajes de la conversación y qué tres condiciones terminan el bucle?

<details>
<summary>Ver respuesta</summary>

Roles: `system`, `user`, `assistant` (con `tool_calls`), `tool` (resultado). Terminan el bucle: respuesta sin `tool_calls` (`finish_reason=stop`), agotar `max_steps`, o error no recuperable (p. ej. `llm_error` persistente).

</details>

---
# 10 · Referencias

- Erik Schluntz y Barry Zhang (Anthropic), *Building effective agents* (dic-2024; autoría y existencia confirmadas por búsqueda web, el dominio no es accesible desde el entorno de revisión; el día exacto «19» no se pudo confirmar) — <https://www.anthropic.com/engineering/building-effective-agents> (definiciones de *workflow* vs *agent*, condiciones de parada, coste y errores acumulados).
- Yao, Zhao, Yu, Du, Shafran, Narasimhan, Cao, *ReAct: Synergizing Reasoning and Acting in Language Models*, 2022 — <https://arxiv.org/abs/2210.03629>.
- Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, NeurIPS 2020 — <https://arxiv.org/abs/2005.11401> (título, primer autor y congreso confirmados por búsqueda; el identificador arXiv no se pudo abrir desde aquí).
- Ji et al., *Survey of Hallucination in Natural Language Generation*, 2022 (arXiv; revisiones posteriores) — <https://arxiv.org/abs/2202.03629>.
- Anthropic, *Tool use with Claude* (documentación) — <https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview> (flujo `tool_use` → `tool_result`, `stop_reason`).
- OpenAI, *Function calling* (guía) — <https://developers.openai.com/docs/guides/function-calling>. (Nuestro adaptador usa Chat Completions con `tools`; la guía actual puede centrarse en otra API: revisa la documentación vigente.)
- Honeycomb, *So We Shipped an AI Product. Did it Work?* (2 de octubre de 2023) — <https://www.honeycomb.io/blog/we-shipped-ai-product>.
- Anthropic, *Pricing / Prompt caching* (multiplicadores de lectura y escritura de caché citados en 🧠 #10: **fuente secundaria, no verificada** contra la página oficial desde este entorno).
- OpenTelemetry, *GenAI semantic conventions* (estado Development; repositorio actual) — <https://github.com/open-telemetry/semantic-conventions-genai>.

**Siguiente:** `02_tres_pilares` — traces, metrics y logs, el modelo de datos de OpenTelemetry y las *golden signals* aplicadas a agentes. Haz primero el **proyecto** de este módulo: `01_proyecto_agente_base.ipynb`.